# IMDB RNN Sentiment

Train a simple RNN model on the IMDB movie review dataset, save the model and preprocessing artifacts, and run a quick prediction sanity check.

In [ ]:
%pip install numpy tensorflow

In [1]:
import json
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

NUM_WORDS = 10000
MAX_LEN = 250
INDEX_FROM = 3
ARTIFACT_DIR = "artifacts"
MODEL_PATH = os.path.join(ARTIFACT_DIR, "imdb_rnn.keras")
WORD_INDEX_PATH = os.path.join(ARTIFACT_DIR, "word_index.json")
META_PATH = os.path.join(ARTIFACT_DIR, "meta.json")

os.makedirs(ARTIFACT_DIR, exist_ok=True)

ModuleNotFoundError: No module named 'numpy'

In [ ]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=NUM_WORDS, index_from=INDEX_FROM)

x_train = pad_sequences(x_train, maxlen=MAX_LEN, padding="post", truncating="post")
x_test = pad_sequences(x_test, maxlen=MAX_LEN, padding="post", truncating="post")

raw_word_index = imdb.get_word_index()
word_index = {k: (v + INDEX_FROM) for k, v in raw_word_index.items()}
word_index["<PAD>"] = 0
word_index["<START>"] = 1
word_index["<OOV>"] = 2

with open(WORD_INDEX_PATH, "w", encoding="utf-8") as f:
    json.dump(word_index, f)

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump({"num_words": NUM_WORDS, "max_len": MAX_LEN}, f)

x_train.shape, x_test.shape

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(NUM_WORDS, 128, input_length=MAX_LEN),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    x_train,
    y_train,
    epochs=3,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

model.save(MODEL_PATH)
MODEL_PATH

In [ ]:
def encode_review(text, word_index, num_words):
    words = text.lower().split()
    encoded = [word_index["<START>"]]
    for word in words:
        idx = word_index.get(word, word_index["<OOV>"])
        if idx >= num_words:
            idx = word_index["<OOV>"]
        encoded.append(idx)
    return encoded

sample_text = "This movie was fantastic, I loved the story and acting."
sample_encoded = encode_review(sample_text, word_index, NUM_WORDS)
sample_padded = pad_sequences([sample_encoded], maxlen=MAX_LEN, padding="post", truncating="post")
score = float(model.predict(sample_padded, verbose=0)[0][0])
label = "good" if score >= 0.5 else "bad"
score, label